# Chronos-2 Day-Ahead Price Forecasting: DK1 & DK2

This notebook applies **Chronos-2** (zero-shot and LoRA fine-tuned) to day-ahead
electricity price forecasting for the DK1 and DK2 bidding zones, as a
foundation-model comparison point against the `lear_dk1` LEAR benchmark in this
repository.

Five configurations, in order of increasing model adaptation:

1. Zero-shot, DK1 univariate (price history only)
2. Zero-shot, DK1 with covariates (day-ahead load and wind+solar forecasts as
   known-future covariates)
3. Zero-shot, DK1 + DK2 **true multivariate** cross-learning — one joint
   2-variate forecast per day, using both zones' covariates as shared inputs
4. LoRA fine-tuned, DK1 univariate
5. LoRA fine-tuned, DK1 with covariates

**Design decisions fixed by interview before writing this notebook** (see chat
for the full reasoning):

| Decision | Choice |
|---|---|
| Point vs. quantile forecasts | **Point forecasts only** — `quantile_levels=[]` throughout |
| Train/test split | Matches `lear_dk1` exactly: history from `2015-01-07`, test days `2023-04-11`–`2025-04-07` (728 days) — DK1/DK2 day-ahead switched to 15-minute market time units on 2025-04-08, which is why the test period stops there even though the cleaned CSVs run to 2025-09-30 |
| Backtest scope | Full rolling backtest: one 24h-ahead forecast per test day, sliding the context window forward each day (no daily *refitting* — Chronos-2 is zero-shot/frozen-adapter, unlike LEAR's daily recalibration) |
| Context length | Fixed at Chronos-2's max context, 8192 hourly steps (≈341 days), for every forecast day — always available given the 2015 history start |
| Covariate framing | Day-ahead load and wind+solar forecasts are **known-future covariates** (they are themselves forecasts published before gate closure, exactly as `lear_dk1`/LEAR uses them) |
| Cross-learning mechanism | **True multivariate** forecasting (2-variate target), not `cross_learning=True` batch sharing — so DK1's forecast can directly use DK2's covariates as model inputs and vice versa |
| LoRA protocol | Fine-tune once on the training period, freeze the adapter, roll it forward zero-shot across the full test period — no periodic refit |
| Metrics | MAE and rMAE (MAE ÷ MAE of the weekly-seasonal-naive `p_hat_d = p_{d-7}`), computed the same way as `lear_dk1/evaluate.py`, so results sit in the same table as the LEAR benchmark |
| Data source | The cleaned `nordic_baltic_clean_hourly.parquet` panel on Google Drive, projected to per-zone CSVs with `run_lear_from_clean.build_csv` — the same projection `lear_dk1` and the DNN Colab notebook use, so all three models read identical inputs. The panel is already imputed centrally in `data_cleaning.ipynb` (no gaps, `assert_epftoolbox_grid(allow_nan=False)`), so this notebook does not re-impute |
| Secrets | The Hugging Face token lives in the project's gitignored `.env` file (`HF_TOKEN=...`, unquoted), read the same way `entsoe_tp/client.py` reads `ENTSOE_API_TOKEN` |

Set `SMOKE_TEST = True` below to validate the whole pipeline on 5 test days
before committing to the full 728-day × 5-configuration run.

## 1. Setup

Mirrors `notebooks/dnn_colab.ipynb`'s Colab setup: clone the repo, mount
Drive, install what's needed. This notebook is PyTorch-only (Chronos-2), so
unlike the DNN notebook it does **not** call `colab_setup.bootstrap()`
wholesale — that installs `requirements-colab.txt` and reports on
TensorFlow, neither of which this notebook needs. It reuses only
`colab_setup.mount_drive()` and `colab_setup.ensure_epftoolbox()` (the latter
needed transitively: `run_lear_from_clean` imports `run_lear_dk1`, which
imports `epftoolbox.data`/`epftoolbox.evaluation` — not `epftoolbox.models`,
so this stays TensorFlow-free).

In [1]:
import os

# Idempotent: safe to re-run without nesting clones.
if not os.path.exists("colab_setup.py"):
    if not os.path.isdir("EPF_Masters"):
        get_ipython().system(
            "git clone https://github.com/Freddy5445/EPF_Masters.git"
        )
    os.chdir("EPF_Masters")
print("cwd:", os.getcwd())

Cloning into 'EPF_Masters'...
remote: Enumerating objects: 483, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 483 (delta 47), reused 62 (delta 35), pack-reused 396 (from 1)
Receiving objects: 100% (483/483), 14.36 MiB | 4.18 MiB/s, done.
Resolving deltas: 100% (230/230), done.
cwd: /content/EPF_Masters


In [2]:
%pip install -q 'chronos-forecasting[extras]>=2.2' 'pandas' 'matplotlib' 'huggingface_hub'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 133.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 11.6 MB/s eta 0:00:00


In [3]:
import colab_setup

colab_setup.mount_drive()
print("epftoolbox:", colab_setup.ensure_epftoolbox())

import run_lear_from_clean  # the zone-from-panel CSV projection lear_dk1/the DNN notebook use

Mounted at /content/drive
epftoolbox: /content/EPF_Masters/epftoolbox/epftoolbox/__init__.py


/content/EPF_Masters/run_lear_from_clean.py:23: SyntaxWarning: invalid escape sequence '\`'
  trailing ``\`` is not a line continuation and truncates the command.
/content/EPF_Masters/run_lear_dk1.py:6: SyntaxWarning: invalid escape sequence '\`'
  where a trailing ``\`` is not a line continuation and truncates the command.


### Hugging Face token

Add your token to the project's `.env` file (gitignored, never committed) as:

```
HF_TOKEN=hf_your_token_here
```

unquoted, one `KEY=value` per line — same convention `entsoe_tp/client.py`
uses for `ENTSOE_API_TOKEN`. On Colab, a per-account secret (key icon in the
left sidebar, add `HF_TOKEN`) also works and is checked first.

In [ ]:
HF_TOKEN_PATH = "/content/drive/MyDrive/EPF_Masters/HF_token.txt"

with open(HF_TOKEN_PATH, encoding="utf-8") as handle:
    hf_token = handle.read().strip()

os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from chronos import BaseChronosPipeline, Chronos2Pipeline
from chronos.chronos2 import preprocess

# This notebook's runtime budget (5 configurations x up to 728 daily forecasts,
# plus two LoRA fine-tuning runs) assumes a GPU -- the L4 Colab runtime already
# validated with chronos_2_quickstart.ipynb. Fails loud rather than silently
# crawling on CPU.
assert torch.cuda.is_available(), (
    "No CUDA GPU visible. Switch the Colab runtime to a GPU before proceeding "
    "-- this notebook is not sized for CPU inference."
)
device_map = "cuda"

pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2", device_map=device_map
)

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

In [18]:
# --- Paths -------------------------------------------------------------
# Same Drive layout notebooks/dnn_colab.ipynb uses, so the parquet only needs
# uploading once for every model in this project.
DRIVE = "/content/drive/MyDrive/EPF_Masters"
PANEL = os.path.join(DRIVE, "nordic_baltic_clean_hourly.parquet")
DATASETS = os.path.join(DRIVE, "datasets")
RESULTS_DIR = Path(DRIVE) / "experiments" / "chronos2"
os.makedirs(DATASETS, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Split, matching lear_dk1 exactly -----------------------------------
DATA_START = pd.Timestamp("2015-01-07")
TEST_START = pd.Timestamp("2023-04-11")
TEST_END = pd.Timestamp("2025-04-07")  # inclusive; DK1/DK2 covariates go to 15-min MTU on 2025-04-08

# --- Forecast / context ---------------------------------------------------
PREDICTION_LENGTH = 24          # hours per forecast day
CONTEXT_LENGTH = 8192           # Chronos-2's max context, ~341 days of hourly history

# --- LoRA fine-tuning (starting points from the Chronos-2 quickstart; not
# tuned for this dataset -- watch the validation loss and adjust) ----------
VALIDATION_DAYS = 60            # trailing slice of the training period held out for val loss
NUM_STEPS = 1000
LEARNING_RATE_LORA = 1e-4
BATCH_SIZE_FT = 32

# --- Smoke test ------------------------------------------------------------
# Validates the full pipeline (data, batching, checkpointing, scoring) on a
# handful of days before committing hours of Colab GPU time.
SMOKE_TEST = True
SMOKE_DAYS = 5

test_days_full = pd.date_range(TEST_START, TEST_END, freq="D")
test_days = test_days_full[:SMOKE_DAYS] if SMOKE_TEST else test_days_full
print(f"{len(test_days_full)} test days total; running {len(test_days)}"
      f"{' (SMOKE_TEST)' if SMOKE_TEST else ''}")

728 test days total; running 5 (SMOKE_TEST)


## Data loading

Same source as every other model in this repo: `nordic_baltic_clean_hourly.parquet`
(written by `data_cleaning.ipynb`, uploaded to Drive), projected to a per-zone
CSV by `run_lear_from_clean.build_csv` — the identical projection
`lear_dk1` and `notebooks/dnn_colab.ipynb` use. The panel is already on naive
local time, 24 rows/day, and fully imputed centrally (`assert_epftoolbox_grid
(allow_nan=False)` is enforced when it's written), so there is nothing left to
impute here, and Chronos-2 is scored against exactly the same "observed"
price LEAR and the DNN are.

In [9]:
if not os.path.exists(PANEL):
    raise FileNotFoundError(
        f"No cleaned panel at {PANEL}. Upload nordic_baltic_clean_hourly.parquet "
        f"(written by data_cleaning.ipynb) to that Drive path first."
    )

zone_csv = {}
for zone in ("DK1", "DK2"):
    path, start, end = run_lear_from_clean.build_csv(
        PANEL, zone, "load-windsolar", DATASETS, f"{zone}_clean_load-windsolar"
    )
    zone_csv[zone] = path


def load_zone(zone: str) -> pd.DataFrame:
    df = pd.read_csv(zone_csv[zone])
    df.columns = ["timestamp", "price", "load_forecast", "windsolar_forecast"]
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_index().asfreq("h")
    df = df.loc[df.index >= DATA_START]

    remaining_na = df.isna().sum()
    if remaining_na.any():
        raise ValueError(
            f"{zone}: unexpected NaN after projecting from the cleaned panel "
            f"(should be none): {remaining_na.to_dict()}"
        )
    return df


dk1 = load_zone("DK1")
dk2 = load_zone("DK2")
print("DK1", dk1.index.min(), "->", dk1.index.max(), f"({len(dk1)} hours)")
print("DK2", dk2.index.min(), "->", dk2.index.max(), f"({len(dk2)} hours)")

DK1: 85,440 hours, 2016-01-02 to 2025-09-30 (3,560 days), 3 columns
written: /content/drive/MyDrive/EPF_Masters/datasets/DK1_clean_load-windsolar.csv

DK2: 85,440 hours, 2016-01-02 to 2025-09-30 (3,560 days), 3 columns
written: /content/drive/MyDrive/EPF_Masters/datasets/DK2_clean_load-windsolar.csv

DK1 2016-01-02 00:00:00 -> 2025-09-30 23:00:00 (85440 hours)
DK2 2016-01-02 00:00:00 -> 2025-09-30 23:00:00 (85440 hours)


## Metrics: MAE and rMAE

Matches `lear_dk1/evaluate.py`: MAE against the observed (unimputed) price,
excluding hours with no observed price, divided by the MAE of the
weekly-seasonal-naive forecast `p_hat_d = p_{d-7}` computed **within the test
period only** (so the naive baseline itself gets no benefit from training-period
history, and the first 7 days of the test period are excluded from the naive
MAE, exactly as upstream does).

In [10]:
def naive_weekly_mae(real: pd.Series) -> float:
    diffs = (real - real.shift(24 * 7)).abs()
    return float(diffs.mean())


def score(real: pd.Series, pred: pd.Series, naive_mae: float) -> dict:
    aligned = pd.concat([real.rename("real"), pred.rename("pred")], axis=1)
    aligned = aligned.dropna(subset=["real"])
    mae = float((aligned["real"] - aligned["pred"]).abs().mean())
    return {
        "mae": mae,
        "rmae": mae / naive_mae,
        "hours_scored": int(aligned["real"].notna().sum()),
        "hours_total": int(len(real)),
    }


def test_period_series(df: pd.DataFrame, col: str, days: pd.DatetimeIndex) -> pd.Series:
    start, end = days[0], days[-1] + pd.Timedelta(hours=23)
    return df.loc[start:end, col]


naive_mae_dk1 = naive_weekly_mae(test_period_series(dk1, "price", test_days_full))
naive_mae_dk2 = naive_weekly_mae(test_period_series(dk2, "price", test_days_full))
print(f"Weekly-naive MAE over the full test period -- DK1: {naive_mae_dk1:.3f}, "
      f"DK2: {naive_mae_dk2:.3f}")

Weekly-naive MAE over the full test period -- DK1: 38.875, DK2: 41.392


## Rolling backtest infrastructure

For each test day, one *item* is built: its own trailing `CONTEXT_LENGTH`-hour
context slice, plus (when covariates are used) the known-future covariate
values for that day's 24 hours. All items for a chunk of days are stacked into
one long-format dataframe and forecast in a single `predict_df` call —
vectorised batch inference rather than one Python-level call per day.

Progress is checkpointed to CSV after every chunk (`RESULTS_DIR/<name>.csv`),
so a Colab disconnect partway through a 728-day run loses at most one chunk;
re-running the same cell resumes from the days already scored.

In [11]:
def make_rolling_frames(df, days, target_cols, covariate_cols, context_length, id_prefix):
    """Long-format (context_df, future_df) spanning `days` as separate items.

    `df` must be indexed by hourly timestamp (name 'timestamp') and contain
    every column in target_cols + covariate_cols.
    """
    context_parts, future_parts = [], []
    for day in days:
        day_start = pd.Timestamp(day)
        day_end = day_start + pd.Timedelta(hours=23)
        item_id = f"{id_prefix}{day_start:%Y%m%d}"

        hist = df.loc[:day_start - pd.Timedelta(hours=1)].tail(context_length)
        ctx = hist[target_cols + covariate_cols].reset_index()
        ctx.insert(0, "item_id", item_id)
        context_parts.append(ctx)

        if covariate_cols:
            fut = df.loc[day_start:day_end, covariate_cols].reset_index()
            fut.insert(0, "item_id", item_id)
            future_parts.append(fut)

    context_df = pd.concat(context_parts, ignore_index=True)
    future_df = pd.concat(future_parts, ignore_index=True) if future_parts else None
    return context_df, future_df


def _chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


def _done_dates(path: Path) -> set:
    if not path.exists():
        return set()
    done = pd.read_csv(path, usecols=["timestamp"], parse_dates=["timestamp"])
    return set(done["timestamp"].dt.date.unique())


def run_backtest(name, df, days, target_cols, covariate_cols, model_pipeline,
                  context_length=CONTEXT_LENGTH, prediction_length=PREDICTION_LENGTH,
                  chunk_days=50, batch_size=25, results_dir=RESULTS_DIR):
    path = results_dir / f"{name}.csv"
    done = _done_dates(path)
    remaining = [d for d in days if d.date() not in done]
    print(f"[{name}] {len(done)} day(s) already checkpointed, "
          f"{len(remaining)} remaining")

    header_needed = not path.exists()
    target_arg = target_cols[0] if len(target_cols) == 1 else target_cols

    for chunk in _chunked(remaining, chunk_days):
        context_df, future_df = make_rolling_frames(
            df, chunk, target_cols, covariate_cols, context_length, id_prefix=f"{name}_"
        )
        pred = model_pipeline.predict_df(
            context_df,
            future_df=future_df,
            prediction_length=prediction_length,
            quantile_levels=[],  # point forecasts only
            id_column="item_id",
            timestamp_column="timestamp",
            target=target_arg,
            batch_size=batch_size,
        )
        pred_wide = (
            pred.pivot_table(index=["item_id", "timestamp"], columns="target_name",
                              values="predictions")
            .reset_index()
        )
        pred_wide.to_csv(path, mode="a", header=header_needed, index=False)
        header_needed = False
        print(f"[{name}] +{len(chunk)} day(s) -> {chunk[0].date()}..{chunk[-1].date()}")

    return pd.read_csv(path, parse_dates=["timestamp"])

## 1. Zero-shot: DK1 univariate

In [12]:
res1 = run_backtest(
    name="dk1_zeroshot_univariate",
    df=dk1, days=test_days,
    target_cols=["price"], covariate_cols=[],
    model_pipeline=pipeline,
)
pred1 = res1.set_index("timestamp")["price"].sort_index()
real1 = test_period_series(dk1, "price", test_days)
scores1 = score(real1, pred1, naive_weekly_mae(real1))
print("DK1 zero-shot univariate:", scores1)

[dk1_zeroshot_univariate] 5 day(s) already checkpointed, 0 remaining
DK1 zero-shot univariate: {'mae': 21.183979200833335, 'rmae': nan, 'hours_scored': 120, 'hours_total': 120}


## 2. Zero-shot: DK1 with covariates

In [13]:
res2 = run_backtest(
    name="dk1_zeroshot_covariates",
    df=dk1, days=test_days,
    target_cols=["price"], covariate_cols=["load_forecast", "windsolar_forecast"],
    model_pipeline=pipeline,
)
pred2 = res2.set_index("timestamp")["price"].sort_index()
scores2 = score(real1, pred2, naive_weekly_mae(real1))
print("DK1 zero-shot with covariates:", scores2)

[dk1_zeroshot_covariates] 5 day(s) already checkpointed, 0 remaining
DK1 zero-shot with covariates: {'mae': 16.711832748333332, 'rmae': nan, 'hours_scored': 120, 'hours_total': 120}


## 3. Zero-shot: DK1 + DK2 cross-learning (true multivariate), covariates from both

One joint forecast call per day predicts `DK1_price` and `DK2_price` together
as a 2-variate target, with both zones' load and wind+solar forecasts as
shared known-future covariate inputs — so DK1's forecast can directly draw on
DK2's covariates and vice versa. Context is shorter and batch size smaller
here than sections 1-2, since each item now carries two full-length target
series.

In [14]:
wide = dk1[["price", "load_forecast", "windsolar_forecast"]].rename(columns={
    "price": "DK1_price", "load_forecast": "DK1_load_forecast",
    "windsolar_forecast": "DK1_windsolar_forecast",
}).join(dk2[["price", "load_forecast", "windsolar_forecast"]].rename(columns={
    "price": "DK2_price", "load_forecast": "DK2_load_forecast",
    "windsolar_forecast": "DK2_windsolar_forecast",
}), how="inner")

target_cols_mv = ["DK1_price", "DK2_price"]
covariate_cols_mv = ["DK1_load_forecast", "DK1_windsolar_forecast",
                     "DK2_load_forecast", "DK2_windsolar_forecast"]

res3 = run_backtest(
    name="dk1dk2_zeroshot_multivariate",
    df=wide, days=test_days,
    target_cols=target_cols_mv, covariate_cols=covariate_cols_mv,
    model_pipeline=pipeline,
    chunk_days=25, batch_size=8,  # heavier per-item: two full-length target series
)

pred3_dk1 = res3.set_index("timestamp")["DK1_price"].sort_index()
pred3_dk2 = res3.set_index("timestamp")["DK2_price"].sort_index()
real2 = test_period_series(dk2, "price", test_days)

scores3_dk1 = score(real1, pred3_dk1, naive_weekly_mae(real1))
scores3_dk2 = score(real2, pred3_dk2, naive_weekly_mae(real2))
print("DK1 (multivariate):", scores3_dk1)
print("DK2 (multivariate):", scores3_dk2)

[dk1dk2_zeroshot_multivariate] 5 day(s) already checkpointed, 0 remaining
DK1 (multivariate): {'mae': 16.70089224166667, 'rmae': nan, 'hours_scored': 120, 'hours_total': 120}
DK2 (multivariate): {'mae': 22.564021108333336, 'rmae': nan, 'hours_scored': 120, 'hours_total': 120}


## 4. LoRA fine-tuned: DK1 univariate

Fine-tuned once on the training period (`DATA_START` → `TEST_START`), adapter
frozen, then rolled forward zero-shot across the test period exactly like
section 1 — no periodic refitting.

In [19]:
# Colab ships an older torchao than peft now requires for LoRA (0.10.0 vs.
# the >0.16.0 peft's torchao dispatcher checks for) -- same fix and same
# placement (right before the first LoRA fit) as chronos_2_quickstart.ipynb.
%pip install -q --upgrade torchao

KeyboardInterrupt: 

In [20]:
train_df_dk1_uni = (
    dk1.loc[dk1.index < TEST_START, ["price"]]
    .reset_index()
)
train_df_dk1_uni.insert(0, "item_id", "DK1")

val_cutoff = TEST_START - pd.Timedelta(days=VALIDATION_DAYS)
train_inputs_uni = preprocess.from_data_frame(
    train_df_dk1_uni[train_df_dk1_uni["timestamp"] < val_cutoff],
    target_columns=["price"],
    prediction_length=PREDICTION_LENGTH,
    id_column="item_id",
    timestamp_column="timestamp",
)
val_inputs_uni = preprocess.from_data_frame(
    train_df_dk1_uni[train_df_dk1_uni["timestamp"] >= val_cutoff],
    target_columns=["price"],
    prediction_length=PREDICTION_LENGTH,
    id_column="item_id",
    timestamp_column="timestamp",
)

lora_pipeline_uni = pipeline.fit(
    inputs=train_inputs_uni,
    validation_inputs=val_inputs_uni,
    prediction_length=PREDICTION_LENGTH,
    finetune_mode="lora",
    num_steps=NUM_STEPS,
    learning_rate=LEARNING_RATE_LORA,
    batch_size=BATCH_SIZE_FT,
    logging_steps=100,
)

Step,Training Loss,Validation Loss
100,3.330256,14.924036
200,3.355690,15.622475
300,3.285322,15.520527
400,3.253527,15.704000
500,3.185079,16.188290
600,3.149725,17.156284
700,3.268473,16.975830
800,3.236901,17.232107
900,3.094901,17.041977
1000,3.136855,17.205158


In [21]:
res4 = run_backtest(
    name="dk1_lora_univariate",
    df=dk1, days=test_days,
    target_cols=["price"], covariate_cols=[],
    model_pipeline=lora_pipeline_uni,
)
pred4 = res4.set_index("timestamp")["price"].sort_index()
scores4 = score(real1, pred4, naive_weekly_mae(real1))
print("DK1 LoRA univariate:", scores4)

[dk1_lora_univariate] 0 day(s) already checkpointed, 5 remaining
[dk1_lora_univariate] +5 day(s) -> 2023-04-11..2023-04-15
DK1 LoRA univariate: {'mae': 20.2691171875, 'rmae': nan, 'hours_scored': 120, 'hours_total': 120}


## 5. LoRA fine-tuned: DK1 with covariates

Same one-time fine-tune-then-freeze protocol as section 4, with the day-ahead
load and wind+solar forecasts as known-future covariates during both training
and the rolled-forward backtest.

In [22]:
train_df_dk1_cov = (
    dk1.loc[dk1.index < TEST_START, ["price", "load_forecast", "windsolar_forecast"]]
    .reset_index()
)
train_df_dk1_cov.insert(0, "item_id", "DK1")

train_inputs_cov = preprocess.from_data_frame(
    train_df_dk1_cov[train_df_dk1_cov["timestamp"] < val_cutoff],
    target_columns=["price"],
    prediction_length=PREDICTION_LENGTH,
    id_column="item_id",
    timestamp_column="timestamp",
    known_covariates_names=["load_forecast", "windsolar_forecast"],
)
val_inputs_cov = preprocess.from_data_frame(
    train_df_dk1_cov[train_df_dk1_cov["timestamp"] >= val_cutoff],
    target_columns=["price"],
    prediction_length=PREDICTION_LENGTH,
    id_column="item_id",
    timestamp_column="timestamp",
    known_covariates_names=["load_forecast", "windsolar_forecast"],
)

lora_pipeline_cov = pipeline.fit(
    inputs=train_inputs_cov,
    validation_inputs=val_inputs_cov,
    prediction_length=PREDICTION_LENGTH,
    finetune_mode="lora",
    num_steps=NUM_STEPS,
    learning_rate=LEARNING_RATE_LORA,
    batch_size=BATCH_SIZE_FT,
    logging_steps=100,
)

Step,Training Loss,Validation Loss
100,0.872644,1.987155
200,0.864578,1.866869
300,0.813237,1.705624
400,0.814383,2.049835
500,0.834296,1.914562
600,0.830081,1.836851
700,0.829839,2.042682
800,0.805847,1.918653
900,0.785274,1.950590
1000,0.812541,1.998792


In [23]:
res5 = run_backtest(
    name="dk1_lora_covariates",
    df=dk1, days=test_days,
    target_cols=["price"], covariate_cols=["load_forecast", "windsolar_forecast"],
    model_pipeline=lora_pipeline_cov,
)
pred5 = res5.set_index("timestamp")["price"].sort_index()
scores5 = score(real1, pred5, naive_weekly_mae(real1))
print("DK1 LoRA with covariates:", scores5)

[dk1_lora_covariates] 0 day(s) already checkpointed, 5 remaining
[dk1_lora_covariates] +5 day(s) -> 2023-04-11..2023-04-15
DK1 LoRA with covariates: {'mae': 17.090770833333334, 'rmae': nan, 'hours_scored': 120, 'hours_total': 120}


## Summary

To compare against the LEAR benchmark, pull the corresponding ensemble MAE /
rMAE from `lear_dk1`'s own output (`experiments/<run-name>/evaluation.json`,
`scores.ensemble`) and add it as a row below — this notebook only computes the
Chronos-2 side, on the identical test period and identical MAE/rMAE
definition.

In [25]:
summary = pd.DataFrame([
    {"model": "Chronos-2 zero-shot, DK1 univariate", **scores1},
    {"model": "Chronos-2 zero-shot, DK1 + covariates", **scores2},
    {"model": "Chronos-2 zero-shot, DK1 (multivariate w/ DK2)", **scores3_dk1},
    {"model": "Chronos-2 zero-shot, DK2 (multivariate w/ DK1)", **scores3_dk2},
    {"model": "Chronos-2 LoRA, DK1 univariate", **scores4},
    {"model": "Chronos-2 LoRA, DK1 + covariates", **scores5},
]).set_index("model")
summary

,mae,rmae,hours_scored,hours_total
model,,,,
"Chronos-2 zero-shot, DK1 univariate",21.183979,NaN,120,120
"Chronos-2 zero-shot, DK1 + covariates",16.711833,NaN,120,120
"Chronos-2 zero-shot, DK1 (multivariate w/ DK2)",16.700892,NaN,120,120
"Chronos-2 zero-shot, DK2 (multivariate w/ DK1)",22.564021,NaN,120,120
"Chronos-2 LoRA, DK1 univariate",20.269117,NaN,120,120
"Chronos-2 LoRA, DK1 + covariates",17.090771,NaN,120,120
